# 03 — Data Preprocessing

This notebook focuses on preparing the climate dataset for machine learning model development.

The preprocessing workflow will address data quality, missing values, temporal validation, feature selection, outlier investigation, feature engineering, and feature scaling.

All transformations will be designed to prevent data leakage and maintain consistency between training, validation, and testing datasets.

## 3.1 Environment Setup and Data Loading

This section configures the project environment and loads the integrated datasets generated during the dataset integration stage.

Existing reusable functions from the project source modules will be inspected and reused to maintain a modular and consistent preprocessing workflow.

### 3.1.1 Project Environment Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project configuration
from config import (
    TRAIN_DIR,
    TEST_DIR,
    DATA_DIR,
    MODELS_DIR,
    REPORTS_DIR
)

# Third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Existing project functions
from src.data_loader import (
    load_and_standardize_datasets,
    merge_datasets
)

print("Project root:", PROJECT_ROOT)
print("Environment setup completed.")

Project root: D:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge
Environment setup completed.


#### 3.1.2 Load Integrated Datasets

The standardized climate datasets will be loaded and merged using the reusable functions implemented in the project source module.

The original observations will remain unchanged at this stage. Preprocessing operations will be performed in subsequent sections.

In [2]:
# Load and standardize training datasets
train_datasets = load_and_standardize_datasets(TRAIN_DIR)

# Load and standardize testing datasets
test_datasets = load_and_standardize_datasets(TEST_DIR)

# Merge climate features
train_merged = merge_datasets(train_datasets)
test_merged = merge_datasets(test_datasets)

print("Training dataset shape:", train_merged.shape)
print("Testing dataset shape:", test_merged.shape)

Training dataset shape: (2214, 18)
Testing dataset shape: (738, 17)



## 3.2 Data Quality Assessment

This section examines data quality issues that require further investigation before model development.

The assessment focuses on missing value patterns and their potential impact on preprocessing and model evaluation.

Previously completed structural validation checks, including duplicate date verification and dataset alignment, are not repeated in this notebook.


### 3.2.1 Missing Values Investigation

This subsection examines missing values in the training and testing datasets.

The number and percentage of missing values are calculated for each feature to identify incomplete variables and determine the extent of missing data.

No imputation or deletion is performed at this stage.

In [3]:
# Calculate missing values in training data
train_missing = pd.DataFrame({
    "Missing Count": train_merged.isnull().sum(),
    "Missing Percentage (%)": (
        train_merged.isnull().mean() * 100
    )
})

# Calculate missing values in testing data
test_missing = pd.DataFrame({
    "Missing Count": test_merged.isnull().sum(),
    "Missing Percentage (%)": (
        test_merged.isnull().mean() * 100
    )
})

print("TRAINING DATASET — Missing Values")
display(train_missing.round(2))

print("TESTING DATASET — Missing Values")
display(test_missing.round(2))

TRAINING DATASET — Missing Values


,Missing Count,Missing Percentage (%)
date,0,0.0
5cm_soli_moist,0,0.0
mean_dew_point_temp,0,0.0
max_temp,0,0.0
sea_level_pressure,0,0.0
dryspell_warn_7d,0,0.0
sst_cameroon_mean_temperature_kelvin,0,0.0
sst_cameroon_mean_temperature_celsius,0,0.0
sst_cameroon_mean_temperature_uncertainty,0,0.0
sst_cameroon_fraction_of_sea_ice_covered_ocean,0,0.0


TESTING DATASET — Missing Values


,Missing Count,Missing Percentage (%)
date,0,0.00
5cm_soli_moist,0,0.00
mean_dew_point_temp,1,0.14
max_temp,1,0.14
sea_level_pressure,1,0.14
sst_cameroon_mean_temperature_kelvin,123,16.67
sst_cameroon_mean_temperature_celsius,123,16.67
sst_cameroon_mean_temperature_uncertainty,123,16.67
sst_cameroon_fraction_of_sea_ice_covered_ocean,123,16.67
sst_indian_ocean_mean_temperature_kelvin,123,16.67


#### 3.2.1.1 Investigating Missing Value Patterns

This subsection investigates the temporal distribution of missing values in the testing dataset.

The objective is to determine whether missing observations are isolated or concentrated within specific dates and groups of features.

Understanding these patterns is essential before selecting an appropriate missing value treatment strategy.

In [4]:
# Identify dates containing missing values
missing_dates = test_merged.loc[
    test_merged.isnull().any(axis=1),
    ["date"]
].copy()

# Add the number of missing features per date
missing_dates["missing_features"] = (
    test_merged.isnull().sum(axis=1)
    .loc[missing_dates.index]
)

print("Number of dates containing missing values:",
      len(missing_dates))

print("\nMissing dates and affected feature counts:")
display(missing_dates)

# Identify missing date ranges for each feature
for column in test_merged.columns:
    missing_mask = test_merged[column].isnull()

    if missing_mask.any():
        dates = test_merged.loc[missing_mask, "date"]

        print(
            f"{column}: "
            f"{dates.min().date()} to {dates.max().date()} "
            f"({missing_mask.sum()} missing values)"
        )

Number of dates containing missing values: 124

Missing dates and affected feature counts:


,date,missing_features
441,2023-09-11,6
615,2025-07-01,8
616,2025-07-02,8
617,2025-07-03,8
618,2025-07-04,8
...,...,...
733,2025-10-27,8
734,2025-10-28,8
735,2025-10-29,8
736,2025-10-30,8


mean_dew_point_temp: 2023-09-11 to 2023-09-11 (1 missing values)
max_temp: 2023-09-11 to 2023-09-11 (1 missing values)
sea_level_pressure: 2023-09-11 to 2023-09-11 (1 missing values)
sst_cameroon_mean_temperature_kelvin: 2025-07-01 to 2025-10-31 (123 missing values)
sst_cameroon_mean_temperature_celsius: 2025-07-01 to 2025-10-31 (123 missing values)
sst_cameroon_mean_temperature_uncertainty: 2025-07-01 to 2025-10-31 (123 missing values)
sst_cameroon_fraction_of_sea_ice_covered_ocean: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_mean_temperature_kelvin: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_mean_temperature_celsius: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_mean_temperature_uncertainty: 2025-07-01 to 2025-10-31 (123 missing values)
sst_indian_ocean_fraction_of_sea_ice_covered_ocean: 2025-07-01 to 2025-10-31 (123 missing values)
potential_water_deficit: 2023-09-11 to 2023-09-11 (1 missing values)
surface_pressure: 2023-09-11

#### Key Findings

- A total of 124 dates contain missing values in the testing dataset.
- Six numerical features have one missing observation each, all occurring on September 11, 2023.
- All eight SST features have 123 missing observations each.
- SST missing values are concentrated in the continuous period from July 1, 2025, through October 31, 2025.
- The SST missingness pattern indicates a temporal data coverage gap rather than isolated missing observations.
- Missing values originate from two distinct patterns: a single-day data gap and a multi-month SST coverage gap.

No missing values are imputed or removed at this stage. Treatment decisions are deferred until the temporal validation strategy and modeling requirements are established.